<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Laboratorios/15_Lab_GNN_MACE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 15 — GNN y MACE para sistemas físicos

En este laboratorio entrenamos redes de paso de mensajes sobre dos sistemas físicos y usamos un potencial interatómico aprendido, MACE-MP-0:

| Parte | Cuándo | Tiempo | Qué se hace |
|---|---|---|---|
| Ejercicios previos | final de la sesión 4 | 30 min | lectura de un grafo, mini-lotes, grafo de un cristal y MACE como calculador |
| A1 | sesión 5 | 10 min | una GCN clasifica las partículas del coloide del Lab 14 |
| A2 | sesión 5 | 15 min | una red sobre grafos de cristales predice el módulo de compresibilidad |
| B | sesión 5 | 20 min | MACE-MP-0 calcula el módulo de compresibilidad de los mismos cristales |
| C | sesión 5 y después | 15 min en clase | pregunta abierta: diseño e implementación de prueba de una GNN para su tesis |

**Antes de empezar:** active la GPU de Colab. Con el menú en español: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar*. Con el menú en inglés: *Runtime → Change runtime type → T4 GPU → Save*. Las celdas marcadas `# COMPLETAR` son las suyas; cada una viene con una prueba que dice si quedó bien. Los ejercicios previos traen su solución en una celda oculta (*Mostrar código*); úsela solo para comprobar. En la sesión 5, después de una pausa, Colab habrá olvidado lo que se ejecutó en la sesión 4: vuelva a ejecutar la **Preparación** y siga directamente con la parte A1; las partes A y B no dependen de los ejercicios previos. La descarga de los datos de A2 tarda unos minutos: ejecútela al empezar, mientras trabaja en A1.

# Preparación

En Colab, PyTorch ya viene instalado. Instalamos PyTorch Geometric, MACE (que trae ASE) y `spglib`, que identifica el grupo espacial de un cristal. Si Colab pide reiniciar el entorno después de la instalación, hágalo y siga desde la celda de librerías.

In [ ]:
!pip install -q torch_geometric mace-torch spglib

In [ ]:
import gzip
import json
import requests
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
from scipy.spatial import cKDTree
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

import torch
from torch.nn import functional as fn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import from_networkx, to_dense_adj, to_undirected
from torch_geometric.nn import GCNConv, CGConv, global_mean_pool

from ase import Atoms
from ase.build import bulk
from ase.neighborlist import neighbor_list
from ase.eos import EquationOfState
from ase.units import GPa
import spglib

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", dispositivo)

# Ejercicios previos (al final de la sesión 4)

Cuatro ejercicios cortos que preparan las partes A2 y B. Se hacen en clase con el docente; lo que no se termine se completa antes del laboratorio.

## Previo 1. Lectura del grafo: ¿suma o media?

Para predecir una propiedad del grafo completo se reúnen los estados de los nodos con una función invariante: la suma, la media o el máximo. Complete `lectura` y aplíquela a un triángulo y a dos copias de él, con el grado de cada nodo como estado.

In [ ]:
def lectura(h):
    """Suma, media y máximo de los estados h (un arreglo con un valor por nodo)."""
    # COMPLETAR
    raise NotImplementedError("Complete esta función")

In [ ]:
triangulo = nx.cycle_graph(3)
dos_triangulos = nx.disjoint_union(triangulo, triangulo)
for nombre, G in [("un triángulo", triangulo), ("dos triángulos", dos_triangulos)]:
    h = np.array([grado for _, grado in G.degree()], dtype=float)
    s, m, x = lectura(h)
    print(f"{nombre:15s} suma = {s:4.0f}   media = {m:.2f}   máximo = {x:.0f}")

**Responda:** ¿cuál de las tres lecturas distingue un grafo de dos copias de sí mismo? La energía de un cristal es **extensiva**: si se duplica la celda, se duplica. El módulo de compresibilidad es **intensivo**: no cambia. ¿Qué lectura usaría para cada una?

In [ ]:
#@title Solución del previo 1
def lectura(h):
    """Suma, media y máximo de los estados h (un arreglo con un valor por nodo)."""
    return h.sum(), h.mean(), h.max()

for nombre, G in [("un triángulo", triangulo), ("dos triángulos", dos_triangulos)]:
    h = np.array([grado for _, grado in G.degree()], dtype=float)
    s, m, x = lectura(h)
    print(f"{nombre:15s} suma = {s:4.0f}   media = {m:.2f}   máximo = {x:.0f}")
# Solo la suma distingue las dos copias (6 frente a 12): la media y el máximo valen 2 en ambos.
# Energía (extensiva): suma, E = suma de las E_i. Módulo de compresibilidad (intensivo): media.
# Una misma red cristalina se puede describir con una celda primitiva o con una celda más
# grande; una propiedad intensiva no debe depender de esa elección, y la media no depende.

## Previo 2. Mini-lotes: una matriz diagonal por bloques

PyG entrena con varios grafos a la vez uniéndolos en un solo grafo desconectado. Construimos tres grafos pequeños, con el grado como atributo de cada nodo, y los juntamos en un lote con `DataLoader`.

In [ ]:
def k4_menos_arista():
    G = nx.complete_graph(4)
    G.remove_edge(0, 2)
    return G

grafos = [nx.cycle_graph(3), nx.cycle_graph(6), k4_menos_arista()]
lista = []
for G in grafos:
    dato = from_networkx(G)
    dato.x = torch.tensor([[float(G.degree(u))] for u in G.nodes()])
    lista.append(dato)

lote = next(iter(DataLoader(lista, batch_size=3)))
print(lote)
print("batch =", lote.batch.tolist())

Complete la celda siguiente: obtenga la matriz de adyacencia densa del lote con `to_dense_adj(lote.edge_index)[0]` y dibújela con `plt.imshow`; luego calcule la lectura de media de cada grafo con `global_mean_pool(lote.x, lote.batch)` y compruebe que coincide con el grado medio de cada uno.

In [ ]:
# COMPLETAR: matriz del lote, dibujo y lectura de media por grafo

In [ ]:
#@title Solución del previo 2
A_lote = to_dense_adj(lote.edge_index)[0]
plt.figure(figsize=(4, 4))
plt.imshow(A_lote.numpy(), cmap="Greens")
plt.title("adyacencia del mini-lote")
plt.show()

medias = global_mean_pool(lote.x, lote.batch).flatten()
print("media por grafo:", medias.tolist())
print("grado medio:    ", [np.mean([g for _, g in G.degree()]) for G in grafos])
# Tres bloques sobre la diagonal (3, 6 y 4 nodos) y ceros fuera de ellos: los mensajes
# no cruzan de un grafo a otro. Las medias son 2, 2 y 2,5.

## Previo 3. El grafo radial de un cristal

En un cristal, los nodos son los átomos y hay una arista entre dos átomos a menos de un radio de corte $r_c$, contando las **imágenes periódicas**: un átomo cerca del borde de la celda tiene vecinos en la celda de al lado. ASE lo resuelve con `neighbor_list`: `i, j, d = neighbor_list("ijd", atoms, rc)` devuelve una arista dirigida $i\to j$ por cada par de átomos a menos de `rc`, con su distancia `d`.

Usamos el silicio, con estructura de diamante y $a=5{,}431$ Å. Complete `vecinos_por_atomo`.

In [ ]:
si = bulk("Si", "diamond", a=5.431)
print(si, "\nátomos en la celda primitiva:", len(si))

def vecinos_por_atomo(atoms, rc):
    """Número medio de vecinos por átomo a menos de rc (Å), con las imágenes periódicas."""
    # COMPLETAR
    raise NotImplementedError("Complete esta función")

In [ ]:
# Verificación del previo 3
assert vecinos_por_atomo(si, 3.0) == 4, "Cada Si tiene 4 primeros vecinos a 2,35 Å"
assert vecinos_por_atomo(si, 4.0) == 16, "Con 12 segundos vecinos a 3,84 Å suman 16"
print("Previo 3 correcto")

In [ ]:
#@title Solución del previo 3
def vecinos_por_atomo(atoms, rc):
    """Número medio de vecinos por átomo a menos de rc (Å), con las imágenes periódicas."""
    i = neighbor_list("i", atoms, rc)
    return len(i) / len(atoms)

assert vecinos_por_atomo(si, 3.0) == 4
assert vecinos_por_atomo(si, 4.0) == 16
print("Previo 3 correcto")

In [ ]:
radios = np.linspace(2.0, 6.0, 200)
plt.figure(figsize=(5, 3.2))
plt.plot(radios, [vecinos_por_atomo(si, r) for r in radios])
plt.xlabel("$r_c$ (Å)")
plt.ylabel("vecinos por átomo")
plt.title("silicio: capas de vecinos")
plt.show()

La celda primitiva tiene solo 2 átomos, pero con $r_c=4$ Å cada uno tiene 16 vecinos: casi todos son **imágenes periódicas**. Por eso el grafo de un cristal puede tener varias aristas entre los mismos dos átomos de la celda, una por cada imagen. En la parte A2 cada cristal será un grafo así, con $r_c=5$ Å.

## Previo 4. MACE-MP-0 como calculador de ASE

`mace_mp` carga el modelo fundacional MACE-MP-0 (la primera vez lo descarga) y lo devuelve como un **calculador** de ASE: cualquier estructura que lo tenga asignado entrega energía, fuerzas y esfuerzos. Usamos una supercelda de 64 átomos de silicio con los átomos desplazados al azar 0,05 Å; en el cristal perfecto todas las fuerzas serían cero por simetría.

In [ ]:
from mace.calculators import mace_mp
calc = mace_mp(model="medium", default_dtype="float64", device=dispositivo)

si64 = bulk("Si", "diamond", a=5.431, cubic=True).repeat((2, 2, 2))
si64.rattle(stdev=0.05, seed=0)
si64.calc = calc
E = si64.get_potential_energy()
F = si64.get_forces()
S = si64.get_stress()
print(f"E = {E:.4f} eV  ({E / len(si64):.4f} eV/átomo)")
print(f"fuerza máxima: {np.abs(F).max():.3f} eV/Å")
print("esfuerzo (GPa, notación de Voigt):", np.round(S / GPa, 3))

**Observe:** las tres primeras componentes del esfuerzo, la diagonal, no son cero; son negativas y casi iguales. ¿Por qué, si los átomos solo se desplazaron al azar? Pista: usamos la constante de red experimental, $a=5{,}431$ Å, y MACE-MP-0 aprendió de cálculos con el funcional PBE.

**Banco de simetrías.** La energía debe ser **invariante** y las fuerzas **equivariantes**: al rotar la estructura con $R$, $\mathbf F_i\to R\,\mathbf F_i$; al trasladarla, nada cambia; al permutar los átomos, las fuerzas se permutan con ellos. Complete `banco_de_simetrias`, que devuelve los errores de cada prueba. La función `rotacion` y la forma de rotar una celda periódica ya están escritas: se rotan juntas la celda y las posiciones.

In [ ]:
def rotacion(eje, angulo):
    """Matriz de rotación de `angulo` grados alrededor de `eje` (fórmula de Rodrigues)."""
    u = np.asarray(eje, dtype=float) / np.linalg.norm(eje)
    a = np.radians(angulo)
    K = np.array([[0, -u[2], u[1]], [u[2], 0, -u[0]], [-u[1], u[0], 0]])
    return np.eye(3) + np.sin(a) * K + (1 - np.cos(a)) * K @ K

def evaluar(atoms, calc):
    a = atoms.copy()
    a.calc = calc
    return a.get_potential_energy(), a.get_forces()

def rotar(atoms, R):
    a = atoms.copy()
    a.set_cell(atoms.cell[:] @ R.T, scale_atoms=False)
    a.positions = atoms.positions @ R.T
    return a

def banco_de_simetrias(atoms, calc, R, t, perm):
    """Errores de E y F al rotar con R, trasladar con t y permutar los átomos con perm."""
    # COMPLETAR: use evaluar, rotar, atoms.positions + t y atoms[perm]
    raise NotImplementedError("Complete esta función")

In [ ]:
R = rotacion([1, 2, 3], 37)
t = np.array([0.3, -1.1, 2.0])
perm = np.random.default_rng(0).permutation(len(si64))
errores = banco_de_simetrias(si64, calc, R, t, perm)
for prueba, error in errores.items():
    print(f"{prueba:40s} {error:.1e}")
assert max(errores.values()) < 1e-6, "Algún error es demasiado grande: revise la prueba correspondiente"
print("Previo 4 correcto")

In [ ]:
#@title Solución del previo 4
def banco_de_simetrias(atoms, calc, R, t, perm):
    """Errores de E y F al rotar con R, trasladar con t y permutar los átomos con perm."""
    E0, F0 = evaluar(atoms, calc)
    E1, F1 = evaluar(rotar(atoms, R), calc)
    trasladada = atoms.copy()
    trasladada.positions = atoms.positions + t
    E2, F2 = evaluar(trasladada, calc)
    E3, F3 = evaluar(atoms[perm], calc)
    return {"rotación: |E' - E| (eV)": abs(E1 - E0),
            "rotación: máx |F' - R F| (eV/Å)": np.abs(F1 - F0 @ R.T).max(),
            "traslación: |E' - E| (eV)": abs(E2 - E0),
            "traslación: máx |F' - F| (eV/Å)": np.abs(F2 - F0).max(),
            "permutación: |E' - E| (eV)": abs(E3 - E0),
            "permutación: máx |F' - F[perm]| (eV/Å)": np.abs(F3 - F0[perm]).max()}

errores = banco_de_simetrias(si64, calc, R, t, perm)
for prueba, error in errores.items():
    print(f"{prueba:40s} {error:.1e}")
# Todos los errores quedan en el nivel del redondeo de doble precisión (~1e-10 o menos):
# la simetría está garantizada por la arquitectura, no aprendida de los datos.

# Parte A1 — Nodos en el coloide: una GCN frente a la regresión logística

Volvemos al coloide del Lab 14: un instante con unas 4700 partículas, el grafo de vecinos a menos de $r_c=1{,}41\,d$ y la etiqueta experimental "cristalina" ($N_o\ge8$). En el Lab 14, la regresión logística con rasgos hechos a mano, incluidas las agregaciones $h^{(1)}=A\mathbf d$ y $h^{(2)}=A\,h^{(1)}$, llegó a una exactitud de 0,89 en toda la muestra y 0,68 en la franja de la interfaz.

Ahora la red aprende la agregación: le damos solo dos rasgos **locales**, el grado y el coeficiente de agrupamiento, y dos capas GCN. La celda siguiente carga los datos y prepara todo.

In [ ]:
if not Path("hp02s5bb.txt").exists():
    if not Path("pnas09.zip").exists():
        urlretrieve("https://faculty.college.emory.edu/sites/weeks/data/pnas09.zip", "pnas09.zip")
    ZipFile("pnas09.zip").extractall()

datos = np.loadtxt("hp02s5bb.txt")
f0 = datos[datos[:, 5] == 0]                        # instante t = 0
X, N_o = f0[:, :3], f0[:, 3]
y_nodo = (N_o > 7).astype(int)
n = len(X)
rc = 1.41 * 2.30

pares = np.array(sorted(cKDTree(X).query_pairs(rc)))
G = nx.Graph()
G.add_nodes_from(range(n))
G.add_edges_from(map(tuple, pares))

A = nx.to_scipy_sparse_array(G, nodelist=range(n), format="csr", dtype=float)
grado = np.asarray(A.sum(axis=1)).ravel()
agrup = nx.clustering(G)
agrupamiento = np.array([agrup[u] for u in range(n)])
h1 = A @ grado
h2 = A @ h1
agrup_vecinos = (A @ agrupamiento) / np.maximum(grado, 1)
rasgos_lab14 = np.column_stack([grado, agrupamiento, h1, h2, agrup_vecinos])
rasgos_locales = np.column_stack([grado, agrupamiento])

x = StandardScaler().fit_transform(rasgos_locales)
edge_index = to_undirected(torch.tensor(pares.T, dtype=torch.long), num_nodes=n)
coloide = Data(x=torch.tensor(x, dtype=torch.float), edge_index=edge_index,
               y=torch.tensor(y_nodo, dtype=torch.long))
franja = (X[:, 0] > 27) & (X[:, 0] < 36)            # franja de la interfaz, como en el Lab 14
print(coloide)
print(f"fracción cristalina: {y_nodo.mean():.3f}")

## Tarea A1. La GCN
Complete la clase `GCN`: una capa `GCNConv(entradas, ocultas)`, ReLU, *dropout* con $p=0{,}5$ y una segunda capa `GCNConv(ocultas, 2)` que da un puntaje por clase. Recuerde de la sesión 3 que `GCNConv` ya añade los auto-lazos y la normalización simétrica $\hat A$.

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, entradas, ocultas=32, clases=2):
        super().__init__()
        # COMPLETAR: dos capas GCNConv

    def forward(self, x, edge_index):
        # COMPLETAR: GCNConv -> ReLU -> dropout(p=0.5) -> GCNConv
        raise NotImplementedError("Complete esta clase")

In [ ]:
# Verificación de la tarea A1
salida = GCN(2)(coloide.x, coloide.edge_index)
assert salida.shape == (n, 2), f"La salida debe tener forma ({n}, 2), no {tuple(salida.shape)}"
print("Tarea A1 correcta")

La celda siguiente entrena la GCN con el 30 % de las partículas etiquetadas, la misma separación del Lab 14, y la compara con dos regresiones logísticas: con los dos rasgos locales y con los cinco rasgos del Lab 14. Repite con 5 semillas; tarda menos de un minuto en GPU.

In [ ]:
def entrenar_gcn(semilla, epocas=200):
    torch.manual_seed(semilla)
    tr, te = train_test_split(np.arange(n), train_size=0.3, random_state=semilla, stratify=y_nodo)
    modelo = GCN(2).to(dispositivo)
    datos_gpu = coloide.to(dispositivo)
    tr_t = torch.tensor(tr, device=dispositivo)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(epocas):
        modelo.train()
        optimizador.zero_grad()
        perdida = fn.cross_entropy(modelo(datos_gpu.x, datos_gpu.edge_index)[tr_t], datos_gpu.y[tr_t])
        perdida.backward()
        optimizador.step()
    modelo.eval()
    with torch.no_grad():
        prediccion = modelo(datos_gpu.x, datos_gpu.edge_index).argmax(dim=1).cpu().numpy()
    return prediccion, te

def exactitudes(prediccion, te):
    te_f = te[franja[te]]
    return np.mean(prediccion[te] == y_nodo[te]), np.mean(prediccion[te_f] == y_nodo[te_f])

resultados = {"regresión logística, 2 rasgos locales": [], "regresión logística, 5 rasgos del Lab 14": [],
              "GCN, 2 rasgos locales": []}
for semilla in range(5):
    tr, te = train_test_split(np.arange(n), train_size=0.3, random_state=semilla, stratify=y_nodo)
    for nombre, M in [("regresión logística, 2 rasgos locales", rasgos_locales),
                      ("regresión logística, 5 rasgos del Lab 14", rasgos_lab14)]:
        logistica = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(M[tr], y_nodo[tr])
        resultados[nombre].append(exactitudes(logistica.predict(M), te))
    prediccion_gcn, te_gcn = entrenar_gcn(semilla)
    resultados["GCN, 2 rasgos locales"].append(exactitudes(prediccion_gcn, te_gcn))

print(f"{'modelo':<42} {'toda la muestra':>16} {'interfaz':>10}")
for nombre, valores in resultados.items():
    total, interfaz = np.mean(valores, axis=0)
    print(f"{nombre:<42} {total:>16.3f} {interfaz:>10.3f}")

Los aciertos y errores de la GCN (última semilla) en una lámina de la muestra ($8<z<10$ µm): en azul las partículas cristalinas según los autores, con borde rojo las mal clasificadas.

In [ ]:
lamina = (X[:, 2] > 8) & (X[:, 2] < 10)
error = prediccion_gcn != y_nodo
plt.figure(figsize=(7, 6))
plt.scatter(*X[lamina, :2].T, s=18, c=np.where(y_nodo[lamina] == 1, "tab:blue", "lightgray"),
            edgecolors=np.where(error[lamina], "red", "none"), linewidths=1.2)
plt.xlabel("x (µm)"); plt.ylabel("y (µm)"); plt.gca().set_aspect("equal")
plt.title(f"errores de la GCN en la lámina: {error[lamina].sum()} de {lamina.sum()}")
plt.show()

**Discutan:** con solo dos rasgos locales, ¿alcanza la GCN a la regresión logística con las agregaciones hechas a mano? ¿Qué hacen las dos capas GCN que hacían a mano $h^{(1)}$ y $h^{(2)}$? ¿Por qué la exactitud en la interfaz sigue siendo baja? Piense en qué información usa la etiqueta $N_o$ (direcciones de los enlaces) y cuál ve la GCN. En la figura, ¿hay errores lejos de la interfaz? ¿Dónde están y por qué?

# Parte A2 — Grafos de cristales: el módulo de compresibilidad

Usamos `matbench_log_kvrh` (Dunn et al., 2020): 10 987 cristales de Materials Project con el logaritmo de su **módulo de compresibilidad**, $\log_{10}K_{\text{VRH}}$ (GPa), calculado con DFT a partir del tensor elástico (de Jong et al., 2015). $K_{\text{VRH}}$ es el promedio de Voigt, Reuss y Hill: una medida de cuánto se resiste el cristal a una compresión uniforme.

Para que el entrenamiento quepa en la sesión tomamos 4000 cristales al azar con a lo sumo 30 átomos en la celda. El archivo pesa unos 4 MB. Si la descarga directa falla, use la alternativa con `matminer` que aparece justo después de las celdas de carga.

In [ ]:
URL = "https://ml.materialsproject.org/projects/matbench_log_kvrh.json.gz"
archivo = Path("matbench_log_kvrh.json.gz")
if not archivo.exists():
    respuesta = requests.get(URL, timeout=300)      # con urlretrieve este servidor responde 403
    respuesta.raise_for_status()
    archivo.write_bytes(respuesta.content)
with gzip.open(archivo, "rt") as f:
    crudo = json.load(f)
columnas = crudo["columns"]
filas = crudo["data"]
i_estructura, i_objetivo = columnas.index("structure"), columnas.index("log10(K_VRH)")
print(len(filas), "cristales; columnas:", columnas)

In [ ]:
def a_atoms(estructura):
    """Convierte el diccionario de una estructura de pymatgen en un objeto Atoms de ASE."""
    simbolos = [sitio["species"][0]["element"] for sitio in estructura["sites"]]
    fraccionales = [sitio["abc"] for sitio in estructura["sites"]]
    return Atoms(simbolos, scaled_positions=fraccionales,
                 cell=estructura["lattice"]["matrix"], pbc=True)

orden = np.random.RandomState(0).permutation(len(filas))   # el mismo orden que la ruta con matminer
cristales, log_k = [], []
for k in orden:
    atoms = a_atoms(filas[k][i_estructura])
    if len(atoms) <= 30:
        cristales.append(atoms)
        log_k.append(filas[k][i_objetivo])
    if len(cristales) == 4000:
        break
log_k = np.array(log_k)
print(len(cristales), "cristales; átomos por celda: mediana", int(np.median([len(c) for c in cristales])))
print("ejemplo:", cristales[0].get_chemical_formula(), f"K_VRH = {10 ** log_k[0]:.0f} GPa")

**Alternativa si la descarga directa falla.** Reemplace las dos celdas anteriores por esta, que usa `matminer` (tarda un par de minutos más en instalarse):
```python
!pip install -q matminer
from matminer.datasets import load_dataset
from pymatgen.io.ase import AseAtomsAdaptor

tabla = load_dataset("matbench_log_kvrh").sample(frac=1, random_state=0)
cristales, log_k = [], []
for estructura, valor in zip(tabla["structure"], tabla["log10(K_VRH)"]):
    if len(estructura) <= 30:
        cristales.append(AseAtomsAdaptor.get_atoms(estructura))
        log_k.append(valor)
    if len(cristales) == 4000:
        break
log_k = np.array(log_k)
```

Cada cristal se convierte en un grafo como el del previo 3, con $r_c=5$ Å. El atributo de nodo es el número atómico $Z$; el de arista, la distancia, que la red expandirá en funciones gaussianas. Separamos 80 % para entrenar y 20 % para probar; los índices de prueba se usarán también en la parte B. La construcción tarda alrededor de un minuto.

In [ ]:
rc_cristal = 5.0      # Å

def grafo_cristal(atoms, y, rc=rc_cristal):
    """Grafo radial periódico: Z en los nodos, distancia en las aristas, y = log10(K)."""
    i, j, dist = neighbor_list("ijd", atoms, rc)
    return Data(z=torch.tensor(atoms.numbers, dtype=torch.long),
                edge_index=torch.tensor(np.vstack([j, i]), dtype=torch.long),
                dist=torch.tensor(dist, dtype=torch.float),
                y=torch.tensor([y], dtype=torch.float),
                num_nodes=len(atoms))

grafos_cristal = [grafo_cristal(a, v) for a, v in zip(cristales, log_k)]
idx_entrenamiento, idx_prueba = train_test_split(np.arange(len(grafos_cristal)), test_size=0.2, random_state=0)
entrenamiento = [grafos_cristal[k] for k in idx_entrenamiento]
prueba = [grafos_cristal[k] for k in idx_prueba]
print(len(entrenamiento), "grafos de entrenamiento y", len(prueba), "de prueba;",
      "aristas por átomo (mediana):", int(np.median([g.num_edges / g.num_nodes for g in grafos_cristal])))

## Tarea A2. La red sobre cristales
Complete `RedCristal`. Ya están escritos el **embebido** (un vector aprendido por elemento) y la expansión de las distancias en 40 gaussianas. Falta:

1. en `__init__`, una lista de `capas` capas `CGConv(canales, dim=40, batch_norm=True)` (la convolución de CGCNN, Xie y Grossman, 2018) y una cabeza `Linear(canales, canales) → SiLU → Linear(canales, 1)`;
2. en `forward`, aplicar las capas en orden, `h = capa(h, datos.edge_index, aristas)`, leer cada cristal con `global_mean_pool(h, datos.batch)` y pasar el resultado por la cabeza.

¿Por qué la **media** y no la suma? Recuerde el previo 1.

In [ ]:
class RedCristal(torch.nn.Module):
    def __init__(self, canales=64, capas=3, n_gauss=40):
        super().__init__()
        self.embebido = torch.nn.Embedding(100, canales)
        self.register_buffer("centros", torch.linspace(0.0, rc_cristal, n_gauss))
        # COMPLETAR: self.capas (torch.nn.ModuleList de CGConv) y self.cabeza (torch.nn.Sequential)

    def expandir(self, dist, ancho=0.25):
        """Distancias -> 40 gaussianas centradas entre 0 y rc."""
        return torch.exp(-((dist[:, None] - self.centros[None, :]) / ancho) ** 2)

    def forward(self, datos):
        h = self.embebido(datos.z)
        aristas = self.expandir(datos.dist)
        # COMPLETAR: capas CGConv, lectura de media por cristal y cabeza
        raise NotImplementedError("Complete esta clase")

In [ ]:
# Verificación de la tarea A2
lote_prueba = next(iter(DataLoader(entrenamiento[:8], batch_size=8)))
salida = RedCristal()(lote_prueba)
assert salida.shape == (8,), f"Debe salir un número por cristal: forma (8,), no {tuple(salida.shape)}"
print("Tarea A2 correcta")

La celda siguiente entrena la red 60 épocas con el error cuadrático medio (sesión 4) y reporta el **error absoluto medio** (MAE) en $\log_{10}K$ sobre los cristales de prueba. La línea base predice siempre la media del entrenamiento. Tarda unos 3 minutos en GPU.

In [ ]:
@torch.no_grad()
def predecir(modelo, conjunto):
    modelo.eval()
    salida = [modelo(lote.to(dispositivo)).cpu() for lote in DataLoader(conjunto, batch_size=256)]
    return torch.cat(salida).numpy()

torch.manual_seed(0)
red = RedCristal().to(dispositivo)
optimizador = torch.optim.Adam(red.parameters(), lr=1e-3)
cargador = DataLoader(entrenamiento, batch_size=64, shuffle=True)
y_prueba = log_k[idx_prueba]
for epoca in range(60):
    red.train()
    for lote in cargador:
        lote = lote.to(dispositivo)
        optimizador.zero_grad()
        perdida = fn.mse_loss(red(lote), lote.y)
        perdida.backward()
        optimizador.step()
    if (epoca + 1) % 10 == 0:
        mae = np.mean(np.abs(predecir(red, prueba) - y_prueba))
        print(f"época {epoca + 1:3d}   MAE de prueba = {mae:.3f}")

prediccion_red = predecir(red, prueba)
mae_red = np.mean(np.abs(prediccion_red - y_prueba))
mae_base = np.mean(np.abs(log_k[idx_entrenamiento].mean() - y_prueba))
print(f"\nMAE en log10(K): red = {mae_red:.3f}, línea base = {mae_base:.3f}")
print(f"un MAE de {mae_red:.3f} en log10 es un error típico de un factor {10 ** mae_red:.2f} en K")

In [ ]:
plt.figure(figsize=(4.5, 4.5))
plt.scatter(y_prueba, prediccion_red, s=6, alpha=0.5)
limites = [min(y_prueba.min(), prediccion_red.min()), max(y_prueba.max(), prediccion_red.max())]
plt.plot(limites, limites, "k--", lw=1)
plt.xlabel("log$_{10}K$ DFT"); plt.ylabel("log$_{10}K$ red")
plt.title(f"cristales de prueba, MAE = {mae_red:.3f}")
plt.gca().set_aspect("equal")
plt.show()

**Discutan:** ¿cuánto mejora la red a la línea base? Cambie `global_mean_pool` por `global_add_pool` y vuelva a entrenar: ¿qué pasa y por qué? (Una misma red cristalina se puede describir con celdas de distinto número de átomos.) ¿Qué cristales predice peor la red?

# Parte B — MACE-MP-0 sobre los mismos cristales

La red de A2 aprendió $K$ de los datos de DFT. MACE-MP-0 no ha visto ningún módulo de compresibilidad: aprendió energías, fuerzas y esfuerzos de relajaciones de Materials Project. Si es un buen potencial, debe poder **calcular** $K$ como lo haría DFT, a partir de la curvatura de la energía:
$$
K=V_0\left.\frac{\partial^2E}{\partial V^2}\right|_{V_0}.
$$
Elegimos 10 cristales **cúbicos** del conjunto de prueba, con a lo sumo 6 átomos en la celda. En un cristal cúbico los promedios de Voigt y de Reuss coinciden con el módulo de la ecuación de estado, así que la comparación con $K_{\text{VRH}}$ es directa. Con pocos átomos, además, las posiciones suelen estar fijadas por la simetría y no hace falta relajarlas al comprimir.

In [ ]:
from mace.calculators import mace_mp
calc = mace_mp(model="medium", default_dtype="float64", device=dispositivo)   # el calculador del previo 4

In [ ]:
def grupo_espacial(atoms):
    simbolo = spglib.get_spacegroup((atoms.cell[:], atoms.get_scaled_positions(), atoms.numbers), symprec=1e-3)
    return int(simbolo.split("(")[1].rstrip(")")) if simbolo else 0

elegidos = [k for k in idx_prueba if len(cristales[k]) <= 6 and grupo_espacial(cristales[k]) >= 195][:10]
for k in elegidos:
    print(f"{cristales[k].get_chemical_formula():12s} grupo {grupo_espacial(cristales[k]):3d}   K_VRH = {10 ** log_k[k]:6.1f} GPa")

## Tarea B. La ecuación de estado con MACE
Complete `modulo_mace`: para cada factor de escala $s$, copie el cristal, escale su celda **y** sus átomos (`set_cell(atoms.cell[:] * s, scale_atoms=True)`), asígnele el calculador y guarde el volumen y la energía. Luego ajuste la ecuación de Birch–Murnaghan con `EquationOfState(V, E, eos="birchmurnaghan").fit()`, que devuelve `v0, e0, B` con `B` en eV/Å³, y devuelva `B / GPa`. Si el mínimo `v0` cae fuera del rango de volúmenes, devuelva `np.nan`: el ajuste no sería confiable.

In [ ]:
def modulo_mace(atoms, calc, escalas=np.linspace(0.97, 1.03, 9)):
    """Módulo de compresibilidad (GPa) de la ecuación de estado E(V) calculada con calc."""
    # COMPLETAR
    raise NotImplementedError("Complete esta función")

In [ ]:
# Verificación de la tarea B: el cobre fcc, con K experimental de unos 140 GPa
K_cu = modulo_mace(bulk("Cu", "fcc", a=3.61), calc)
assert 90 < K_cu < 200, f"K del cobre fuera de rango: {K_cu:.0f} GPa"
print(f"Tarea B correcta: K(Cu) con MACE-MP-0 = {K_cu:.0f} GPa")

In [ ]:
posicion = {k: p for p, k in enumerate(idx_prueba)}
K_dft = np.array([10 ** log_k[k] for k in elegidos])
K_red = np.array([10 ** prediccion_red[posicion[k]] for k in elegidos])
K_mace = np.array([modulo_mace(cristales[k], calc) for k in elegidos])

print(f"{'cristal':12s} {'DFT':>8s} {'red A2':>8s} {'MACE':>8s}   (GPa)")
for k, a, b, c in zip(elegidos, K_dft, K_red, K_mace):
    print(f"{cristales[k].get_chemical_formula():12s} {a:8.1f} {b:8.1f} {c:8.1f}")
ok = np.isfinite(K_mace)
print(f"\nerror relativo medio: red = {np.mean(np.abs(K_red / K_dft - 1)):.2f}, "
      f"MACE = {np.mean(np.abs(K_mace[ok] / K_dft[ok] - 1)):.2f}")
print(f"MACE / DFT (mediana): {np.median(K_mace[ok] / K_dft[ok]):.2f}")

In [ ]:
plt.figure(figsize=(4.8, 4.8))
plt.plot(K_dft, K_red, "o", label="red de A2")
plt.plot(K_dft[ok], K_mace[ok], "s", label="MACE-MP-0")
limites = [0, 1.1 * max(K_dft.max(), K_red.max(), np.nanmax(K_mace))]
plt.plot(limites, limites, "k--", lw=1)
plt.xlim(limites); plt.ylim(limites)
plt.xlabel("$K_{\\mathrm{VRH}}$ DFT (GPa)"); plt.ylabel("$K$ predicho (GPa)")
plt.legend(); plt.gca().set_aspect("equal")
plt.show()

**Discutan:** ¿quién se acerca más a DFT, la red entrenada con estos mismos datos o MACE-MP-0, que nunca vio un módulo de compresibilidad? ¿MACE se equivoca al azar o con un sesgo sistemático? Tenga en cuenta que MACE-MP-0 aprendió del mismo nivel de teoría que los datos de referencia (PBE), que sus datos de entrenamiento son relajaciones de cristales, que pasan por volúmenes cercanos al de equilibrio, y que MACE no representa explícitamente el magnetismo. ¿En qué cristales falla más cada modelo? La red de A2 predice un número; MACE calcula una propiedad física a partir de energías, con las que también podría hacer dinámica molecular o calcular fonones.

# Parte C — Pregunta abierta: una GNN para su tesis

Piense en un problema de su tesis, o de su área, que se pueda plantear como aprendizaje sobre grafos. La parte C es individual y tiene dos entregables: el **diseño** de la idea y una **implementación de prueba**. En clase (15 min) se escribe el diseño y se empieza el prototipo; el prototipo se termina después de la sesión, en el plazo que fije el docente.

## C1. Diseño
Responda en la celda siguiente, en unas 10–15 líneas:

1. **El sistema como grafo.** ¿Qué son los nodos y las aristas? ¿Qué atributos tienen? ¿El grafo es fijo o depende de la geometría (como en MACE)?
2. **La tarea.** ¿Es de nodo, de arista o de grafo? ¿Qué se predice y qué lectura usaría (suma, media u otra) y por qué?
3. **Las simetrías.** ¿Qué debe ser invariante o equivariante: permutaciones, rotaciones, traslaciones, otra simetría del problema?
4. **Los datos.** ¿De dónde salen (simulación, experimento, base de datos)? ¿Cuántos hay o podría generar?
5. **La evaluación.** ¿Contra qué línea base compararía, con qué métrica y cómo separaría entrenamiento y prueba sin filtrar información?

*Escriba aquí su diseño.*

## C2. Implementación de prueba
Un prototipo pequeño que muestre que la idea se puede programar; no se espera un modelo terminado. Si todavía no tiene datos, use un conjunto **sintético** pequeño que imite su sistema (por ejemplo, unas decenas de grafos generados con NetworkX o unas estructuras construidas con ASE) y dígalo explícitamente. El prototipo debe:

1. **Construir los grafos** del diseño, con los atributos de nodo y de arista que propuso, como objetos `Data` de PyG (o `Atoms` de ASE si usa MACE).
2. **Definir el modelo** con la tarea y la lectura del diseño, y hacer una pasada hacia adelante sobre un mini-lote.
3. **Verificar una simetría** del diseño: por ejemplo, que la salida no cambia, o se permuta con los nodos, al renombrarlos, como en la sesión 3 y en el previo 4.
4. **Entrenar unas pocas épocas** y comparar con la línea base del diseño, si los datos lo permiten.

Puede partir del código de A1 (tarea de nodo), de A2 (tarea de grafo) o de B (MACE).

In [ ]:
# Implementación de prueba de la parte C

# 1. Datos: los grafos de su sistema (reales o sintéticos), con sus atributos


# 2. Modelo: tarea y lectura del diseño; una pasada hacia adelante sobre un mini-lote


# 3. Simetría: compruebe la invariancia o la equivarianza que propuso


# 4. Entrenamiento corto y comparación con la línea base (si los datos lo permiten)


**Qué funcionó y qué falta.** En 3–5 líneas: qué pudo comprobar con el prototipo, qué problemas encontró y qué haría falta para convertirlo en un estudio de su tesis.

*Escriba aquí.*

# Para la exposición (5 min)

El docente asigna a cada equipo una parte. Prepare **una tabla o figura** y responda:

- **A1.** ¿Mejora la GCN a la regresión logística del Lab 14? ¿Dónde sigue fallando y por qué?
- **A2.** ¿Por qué la lectura de media y no la de suma para $K$? ¿Qué cristales predice peor la red?
- **B.** ¿Concuerda MACE-MP-0 con DFT? ¿Por qué no del todo?

Al final, tres o cuatro estudiantes presentan su propuesta de la parte C en 1 minuto.

# Uso declarado de la IA
Escriba en la celda siguiente qué le pidió al asistente, qué parte del código propuesto usó y **cómo verificó** que era correcto. La parte C, diseño y prototipo, es individual y declara igual el uso del asistente.

*Escriba aquí su declaración.*

**Datos y modelos.**
- Coloide: J. Hernández-Guzmán y E. R. Weeks, «The equilibrium intrinsic crystal–liquid interface of colloids», *Proc. Natl. Acad. Sci. USA* **106**, 15198 (2009), [doi:10.1073/pnas.0904682106](https://doi.org/10.1073/pnas.0904682106).
- Cristales: A. Dunn et al., «Benchmarking materials property prediction methods: the Matbench test set and Automatminer reference algorithm», *npj Comput. Mater.* **6**, 138 (2020), [doi:10.1038/s41524-020-00406-3](https://doi.org/10.1038/s41524-020-00406-3); M. de Jong et al., «Charting the complete elastic properties of inorganic crystalline compounds», *Sci. Data* **2**, 150009 (2015), [doi:10.1038/sdata.2015.9](https://doi.org/10.1038/sdata.2015.9).
- CGCNN: T. Xie y J. C. Grossman, *Phys. Rev. Lett.* **120**, 145301 (2018), [doi:10.1103/PhysRevLett.120.145301](https://doi.org/10.1103/PhysRevLett.120.145301).
- MACE-MP-0: I. Batatia et al., «A foundation model for atomistic materials chemistry», [arXiv:2401.00096](https://arxiv.org/abs/2401.00096).
- Suavizado de los potenciales universales: B. Deng et al., [arXiv:2405.07105](https://arxiv.org/abs/2405.07105).